# Retinal DR Screening — Kaggle pipeline

Runs the whole project on Kaggle: build manifests → cache preprocessed images →
pretrain on EyePACS → fine-tune on APTOS → evaluate on APTOS test and IDRiD →
export the demo model.

**Before running**
1. *Settings → Accelerator*: **GPU T4 x2**. *Settings → Internet*: **On** (to clone the repo and download ImageNet weights).
2. *Add Input*: attach **APTOS 2019 Blindness Detection**, an **EyePACS 2015 / Diabetic Retinopathy Detection** dataset (a pre-resized copy saves hours), and **IDRiD**.
3. Stage 1 takes longer than one interactive session. Use **Save Version → Save & Run All (Commit)** so it runs in the background for up to 12 h, then attach this notebook's output as an input to the next version to resume (see section 4).

In [ ]:
!git clone -q https://github.com/ashwinruke/retina-screen.git
%cd retina-screen
!pip install -q timm albumentations
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Locate the datasets

Kaggle mounts inputs under `/kaggle/input/<slug>`. The guesses below match on
keywords; if one is wrong, set the path by hand.

In [ ]:
from pathlib import Path

inputs = sorted(p for p in Path('/kaggle/input').iterdir())
for p in inputs:
    print(p)

def find(*keywords):
    for p in inputs:
        if any(k in p.name.lower() for k in keywords):
            return p
    return None

APTOS_DIR = find('aptos')
EYEPACS_DIR = find('diabetic-retinopathy', 'eyepacs', 'retinopathy-resized', 'diabetic_retinopathy')
IDRID_DIR = find('idrid', 'indian-diabetic')
print('\nAPTOS  :', APTOS_DIR, '\nEyePACS:', EYEPACS_DIR, '\nIDRiD  :', IDRID_DIR)
assert APTOS_DIR and EYEPACS_DIR and IDRID_DIR, 'Set the missing path(s) by hand'

## 2. Build manifests
APTOS is deduplicated (conflicting-label copies dropped); EyePACS is split by patient; IDRiD keeps its official split.

In [ ]:
!python scripts/prepare_data.py --aptos "{APTOS_DIR}" --eyepacs "{EYEPACS_DIR}" --idrid "{IDRID_DIR}" --out-dir data/manifests

## 3. Cache preprocessed images

Crop → 512 px → Ben Graham → circular mask, once, saved as JPEG. APTOS and IDRiD
take minutes; EyePACS takes roughly an hour from full-size images and much less
from a pre-resized copy. Re-running skips finished images.

**Tip:** after caching EyePACS, save a version of this notebook. Its output
(`cache/`) can then be attached as an input to later versions so you never
recompute it.

In [ ]:
!python scripts/cache_images.py --manifest data/manifests/aptos.csv   --out-dir cache --config configs/finetune_aptos.yaml --workers 4
!python scripts/cache_images.py --manifest data/manifests/idrid.csv   --out-dir cache --config configs/finetune_aptos.yaml --workers 4
!python scripts/cache_images.py --manifest data/manifests/eyepacs.csv --out-dir cache --config configs/pretrain_eyepacs.yaml --workers 4

In [ ]:
import matplotlib.pyplot as plt, matplotlib.image as mpimg
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, name in zip(axes, ['aptos', 'eyepacs', 'idrid']):
    ax.imshow(mpimg.imread(f'cache/{name}_preview.jpg')); ax.set_title(name); ax.axis('off')
plt.show()

## 4. Stage 1 — pretrain on EyePACS

If this is a continuation run, attach the previous version's output as an input,
copy `last.pt` back and resume:

```python
!mkdir -p outputs/pretrain_eyepacs
!cp /kaggle/input/<previous-version>/retina-screen/outputs/pretrain_eyepacs/*.pt outputs/pretrain_eyepacs/
!python scripts/train.py --config configs/pretrain_eyepacs.yaml --resume outputs/pretrain_eyepacs/last.pt
```

In [ ]:
!python scripts/train.py --config configs/pretrain_eyepacs.yaml

## 5. Stage 2 — fine-tune on APTOS (main result, A1)

In [ ]:
!python scripts/train.py --config configs/finetune_aptos.yaml

## 6. Evaluate
Thresholds are fit on APTOS **val** and frozen. Then APTOS test (in-distribution) and all of IDRiD (external).

In [ ]:
!python scripts/evaluate.py --checkpoint outputs/finetune_aptos/best.pt \
    --manifest cache/aptos.csv --splits test \
    --fit-manifest cache/aptos.csv --fit-splits val \
    --name A1_aptos_test --export outputs/model.pt
!python scripts/evaluate.py --checkpoint outputs/model.pt \
    --manifest cache/idrid.csv --splits all --name A1_idrid

## 7. Ablations

Each one changes a single thing relative to A1. Skip any you don't have quota for.

In [ ]:
# A0: no EyePACS pretraining
!python scripts/train.py --config configs/baseline_aptos.yaml
!python scripts/evaluate.py --checkpoint outputs/baseline_aptos/best.pt --manifest cache/aptos.csv --splits test \
    --fit-manifest cache/aptos.csv --fit-splits val --name A0_aptos_test --export outputs/A0.pt
!python scripts/evaluate.py --checkpoint outputs/A0.pt --manifest cache/idrid.csv --splits all --name A0_idrid

In [ ]:
# A2: no Ben Graham (needs a separate cache, and ideally a separate EyePACS pretrain;
# this cheaper version only removes it from fine-tuning, starting from ImageNet)
!python scripts/cache_images.py --manifest data/manifests/aptos.csv --out-dir cache_nobg --no-ben-graham
!python scripts/cache_images.py --manifest data/manifests/idrid.csv --out-dir cache_nobg --no-ben-graham
!python scripts/train.py --config configs/baseline_aptos.yaml experiment.name=A2_nobg data.manifest=cache_nobg/aptos.csv
!python scripts/evaluate.py --checkpoint outputs/A2_nobg/best.pt --manifest cache_nobg/aptos.csv --splits test \
    --fit-manifest cache_nobg/aptos.csv --fit-splits val --name A2_aptos_test --export outputs/A2.pt
!python scripts/evaluate.py --checkpoint outputs/A2.pt --manifest cache_nobg/idrid.csv --splits all --name A2_idrid

In [ ]:
# A3: classification head instead of regression
!python scripts/train.py --config configs/finetune_aptos.yaml experiment.name=A3_cls model.head=classification
!python scripts/evaluate.py --checkpoint outputs/A3_cls/best.pt --manifest cache/aptos.csv --splits test \
    --fit-manifest cache/aptos.csv --fit-splits val --name A3_aptos_test --export outputs/A3.pt
!python scripts/evaluate.py --checkpoint outputs/A3.pt --manifest cache/idrid.csv --splits all --name A3_idrid

In [ ]:
# A4: adapt to IDRiD with its training images, report on the official IDRiD test split only
!python scripts/train.py --config configs/finetune_idrid.yaml
!python scripts/evaluate.py --checkpoint outputs/finetune_idrid/last.pt --manifest cache/idrid.csv --splits test \
    --fit-manifest cache/aptos.csv --fit-splits val --name A4_idrid_test
# Compare against A1 on the same 103 images:
!python scripts/evaluate.py --checkpoint outputs/model.pt --manifest cache/idrid.csv --splits test --name A1_idrid_test

## 8. Results table

In [ ]:
import json, glob, pandas as pd
rows = []
for f in sorted(glob.glob('outputs/eval/*.json')):
    r = json.load(open(f)); t = r['tuned_thresholds']; ref = t['referable']
    rows.append({'run': Path(f).stem, 'n': t['n'], 'QWK': t['qwk'], 'acc': t['accuracy'],
                 'ref_AUC': ref['auc'], 'sens': ref['sensitivity'], 'spec': ref['specificity']})
results = pd.DataFrame(rows).round(3)
print(results.to_markdown(index=False))

## 9. Grad-CAM examples

In [ ]:
import sys, torch, numpy as np, matplotlib.pyplot as plt
sys.path.insert(0, 'src')
from retina.engine import load_for_inference
from retina.data import build_transforms
from retina.gradcam import gradcam, overlay
from retina.preprocess import read_rgb, preprocess_from_config, crop_and_resize
from retina.metrics import scores_to_grades

model, cfg, thr = load_for_inference('outputs/model.pt', 'cpu')
tf = build_transforms(cfg['train']['image_size'], train=False)
df = pd.read_csv('data/manifests/idrid.csv')
sample = pd.concat([g.sample(min(2, len(g)), random_state=0) for _, g in df.groupby('grade')])

cols = (len(sample) + 1) // 2
fig, axes = plt.subplots(2, cols, figsize=(4 * cols, 8))
for ax, (_, row) in zip(axes.T.flatten(), sample.iterrows()):
    raw = read_rgb(row.image_path)  # original, uncached image
    x = tf(image=preprocess_from_config(raw, cfg['preprocess']))['image'][None]
    with torch.no_grad():
        s = float(model.to_score(model(x)))
    ax.imshow(overlay(crop_and_resize(raw, cfg['preprocess']['size']), gradcam(model, x)))
    ax.set_title(f"true {row.grade} | pred {scores_to_grades(np.array([s]), thr)[0]} ({s:.2f})")
    ax.axis('off')
plt.tight_layout(); plt.savefig('outputs/gradcam_idrid.png', dpi=120); plt.show()

## 10. Ship the demo

Download `outputs/model.pt` (or push it to a Hugging Face model repo), then
locally: `bash scripts/build_space.sh model.pt space/` and push `space/` to a
new Gradio Space.